# Capítulo 14 · Laboratório 14: ciclos e sazonalidade
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap14.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O laboratório testa a sazonalidade mensal do Ibovespa com p-valor por bootstrap, o "venda em maio", a virada do mês, o dia da semana e o ciclo presidencial brasileiro; confere a janela de abril de Jake Bernstein e o teste do garimpo; refaz o teste do scanner sazonal no Dow Jones; confere as previsões de ciclo para 2022 da 3ª edição; procura o ciclo dominante do Ibovespa por periodograma; e mede a queda dentro do ano.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap14.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=1: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
DOW = ["AAPL", "AMGN", "AMZN", "AXP", "BA", "CAT", "CRM", "CSCO", "CVX", "DIS", "GS", "HD", "HON", "IBM", "JNJ",
       "JPM", "KO", "MCD", "MMM", "MRK", "MSFT", "NKE", "NVDA", "PG", "SHW", "TRV", "UNH", "V", "VZ", "WMT"]


def baixa(t, ini):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=True, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for nome, (t, ini) in {"Ibovespa": ("^BVSP", "1995-01-01"), "S&P 500": ("^GSPC", "1950-01-01"),
                       "Dow Jones": ("^DJI", "2020-01-01"), "DBC": ("DBC", "2019-01-01"),
                       "Juros 10 anos": ("^TNX", "2019-01-01"), "SBSP3": ("SBSP3.SA", "2018-01-01")}.items():
    if nome not in D:
        D[nome] = baixa(t, ini)
confere({k: D[k] for k in ("Ibovespa", "S&P 500", "Dow Jones", "DBC", "Juros 10 anos", "SBSP3")})
if "Dow30" not in D:
    D["Dow30"] = pd.DataFrame({t: baixa(t, "2010-09-01")["Close"] for t in DOW})
pd.to_pickle(D, CACHE)
rng = np.random.default_rng(42)

## Parte 1: dados

In [ ]:
def fecha(nome):
    c = D[nome]["Close"]
    return c[c > 0]


IB, SPX = fecha("Ibovespa"), fecha("S&P 500")
COMPLETO = IB.loc[:str(IB.index[-1].year - 1)]                 # só anos fechados (o ano corrente distorce médias)
print(f"Parte 1. Ibovespa desde {IB.index[0]:%Y} (anos fechados até {COMPLETO.index[-1]:%Y}), S&P 500 desde"
      f" {SPX.index[0]:%Y}, 30 ações do Dow desde {D['Dow30'].index[0]:%Y}; último pregão {IB.index[-1]:%d/%m/%Y}.")

## Parte 2: sazonalidade mensal com bootstrap

In [ ]:
MENSAL = COMPLETO.resample("ME").last().pct_change().dropna()
MESES = ["janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho", "agosto", "setembro", "outubro",
         "novembro", "dezembro"]
TAB_MES = []
print("\nParte 2. Ibovespa por mês desde 1995 (p-valor por bootstrap; abaixo de 0,05 seria significativo)")
for m in range(1, 13):
    x = MENSAL[MENSAL.index.month == m]
    medias = np.array([rng.choice(MENSAL.values, len(x)).mean() for _ in range(5000)])
    p = (np.abs(medias - MENSAL.mean()) >= abs(x.mean() - MENSAL.mean())).mean()
    TAB_MES.append((MESES[m - 1], x.mean(), (x > 0).mean(), len(x), p))
    print(f"   {MESES[m - 1]:<10} média {ps(x.mean())} | positivos {pc((x > 0).mean(), 0)} | {len(x)} anos | p-valor {nr(p)}")
SP_MES = SPX.loc["1950":"2025"].resample("ME").last().pct_change().dropna()
SP_MES = SP_MES.groupby(SP_MES.index.month).mean()
print("   S&P 500 (1950 a 2025), os quatro melhores meses: " + ", ".join(MESES[m - 1] for m in SP_MES.nlargest(4).index)
      + f"; o pior: {MESES[SP_MES.idxmin() - 1]}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3.5))
plt.bar([m[:3] for m, *_ in TAB_MES], [x[1] * 100 for x in TAB_MES], color="gray")
plt.axhline(0, color="black")
plt.title("Ibovespa: retorno médio de cada mês (1995 a 2025)")
plt.show()

## Parte 3: venda em maio, virada do mês, dia da semana

In [ ]:
NOV_ABR, MAI_OUT = [], []
for a in range(1995, IB.index[-1].year):
    s1 = IB.loc[f"{a}-10-31":f"{a + 1}-04-30"]
    s2 = IB.loc[f"{a + 1}-04-30":f"{a + 1}-10-31"]
    if len(s1) > 50 and len(s2) > 50:
        NOV_ABR.append(s1.iloc[-1] / s1.iloc[0] - 1)
        MAI_OUT.append(s2.iloc[-1] / s2.iloc[0] - 1)
NOV_ABR, MAI_OUT = np.array(NOV_ABR), np.array(MAI_OUT)
DIA = IB.pct_change().dropna().to_frame("r")
DIA["mes"] = DIA.index.to_period("M")
pos = DIA.groupby("mes").cumcount()
tam = DIA.groupby("mes")["r"].transform("size")
VIRADA = (pos <= 2) | (pos == tam - 1)                          # último pregão do mês e os 3 primeiros do seguinte
SEMANA = ["segunda", "terça", "quarta", "quinta", "sexta"]
DSEM = {SEMANA[k]: DIA["r"][DIA.index.dayofweek == k].mean() for k in range(5)}
print(f"\nParte 3. Novembro a abril: média {ps(NOV_ABR.mean())} ({pc((NOV_ABR > 0).mean(), 0)} positivos) | maio a outubro:"
      f" {ps(MAI_OUT.mean())} ({pc((MAI_OUT > 0).mean(), 0)} positivos); {len(NOV_ABR)} anos")
print(f"   virada do mês: {ps(DIA['r'][VIRADA].mean(), 3)} por dia | demais dias: {ps(DIA['r'][~VIRADA].mean(), 3)} por dia")
print("   por dia da semana: " + " | ".join(f"{k} {ps(v, 3)}" for k, v in DSEM.items()))

## Parte 4: ciclo presidencial brasileiro

In [ ]:
ANUAL = COMPLETO.resample("YE").last().pct_change().dropna()
FASES = {1: "1º ano do mandato", 2: "2º ano", 3: "3º ano", 0: "ano da eleição"}
PRES = {}
print("\nParte 4. Ibovespa no ciclo presidencial brasileiro (eleições em 1998, 2002, ..., 2022)")
for k in (1, 2, 3, 0):
    x = ANUAL[(ANUAL.index.year - 1998) % 4 == k]
    PRES[FASES[k]] = x
    print(f"   {FASES[k]:<18} média {ps(x.mean())} | mediana {ps(x.median())} | positivos {pc((x > 0).mean(), 0)} | {len(x)}"
          " anos: " + " ".join(f"{a} {ps(v, 0)}" for a, v in zip(x.index.year, x.values)))

## Parte 5: datas-chave de Bernstein e o garimpo

In [ ]:
def janela(serie, mi, di, mf, df):
    res = []
    for a in sorted(set(serie.index.year)):
        s = serie.loc[f"{a}-{mi:02d}-{di:02d}":f"{a}-{mf:02d}-{df:02d}"]
        if len(s) > 3:
            res.append(s.iloc[-1] / s.iloc[0] - 1)
    return np.array(res)


ABRIL = janela(SPX, 4, 7, 4, 23)
RET = np.log(SPX).diff().dropna()
RET = RET[RET.index.year < SPX.index[-1].year]
ANOS_M = np.vstack([v for v in RET.groupby(RET.index.year).apply(lambda x: x.values[:250]) if len(v) == 250])


def contar_datas(m, dur=12, corte=0.75):
    soma = np.cumsum(m, axis=1)
    n = 0
    for i in range(m.shape[1] - dur):
        r = soma[:, i + dur] - soma[:, i]
        n += max((r > 0).mean(), (r < 0).mean()) >= corte      # vale comprar ou vender
    return n


REAL = contar_datas(ANOS_M)
FALSO = [contar_datas(rng.permuted(ANOS_M, axis=1)) for _ in range(20)]   # embaralha os dias dentro de cada ano
print(f"\nParte 5. S&P 500 de 7 a 23 de abril, {len(ABRIL)} anos: positivo em {pc((ABRIL > 0).mean(), 0)}, média"
      f" {ps(ABRIL.mean(), 2)}")
print(f"   janelas de 12 pregões com acerto de 75% ou mais: {REAL} no S&P real | {nr(np.mean(FALSO), 1)} em média (máximo"
      f" de {max(FALSO)}) em 20 sorteios com os dias embaralhados")

## Parte 6: o teste do tempo

In [ ]:
DJ, DBC, TNX = fecha("Dow Jones"), fecha("DBC"), fecha("Juros 10 anos")
TT = {"dow_max": (DJ.loc[:"2022"].idxmax(), DJ.loc[:"2022"].max()), "dow_min": (DJ.loc["2022"].idxmin(), DJ.loc["2022"].min()),
      "dbc_min": (DBC.loc[:"2020"].idxmin(), DBC.loc[:"2020"].min()), "dbc_max": (DBC.loc["2021":"2023"].idxmax(),
      DBC.loc["2021":"2023"].max()), "dbc_fim23": DBC.loc["2023"].iloc[-1],
      "tnx_min": (TNX.loc[:"2020"].idxmin(), TNX.loc[:"2020"].min()), "tnx_max": (TNX.loc["2021":"2024"].idxmax(),
      TNX.loc["2021":"2024"].max()), "dow_ano22": DJ.loc["2022-02-24"] / DJ.loc[:"2021"].iloc[-1] - 1}
print("\nParte 6. Previsões de ciclo para 2022 citadas na 3ª edição")
print(f"   Dow: máxima em {TT['dow_max'][0]:%d/%m/%Y}, mínima de 2022 em {TT['dow_min'][0]:%d/%m/%Y}, queda de"
      f" {pc(1 - TT['dow_min'][1] / TT['dow_max'][1], 0)}; em 24/02/2022 (fechamento) o Dow caía {pc(-TT['dow_ano22'])} no ano")
print(f"   DBC: mínima em {TT['dbc_min'][0]:%m/%Y}, máxima em {TT['dbc_max'][0]:%m/%Y}, queda de {pc(1 - TT['dbc_fim23'] / TT['dbc_max'][1], 0)}"
      " do topo ao fim de 2023")
print(f"   Juros de 10 anos: mínima de {nr(TT['tnx_min'][1])}% em {TT['tnx_min'][0]:%m/%Y}, máxima de {nr(TT['tnx_max'][1])}% em"
      f" {TT['tnx_max'][0]:%m/%Y}")

## Parte 7: ciclo dominante por periodograma

In [ ]:
SEM_IB = np.log(IB.resample("W").last().dropna())
TEND = np.polyval(np.polyfit(np.arange(len(SEM_IB)), SEM_IB.values, 2), np.arange(len(SEM_IB)))
XC = (SEM_IB.values - TEND - (SEM_IB.values - TEND).mean()) * np.hanning(len(SEM_IB))
POT = np.abs(np.fft.rfft(XC)) ** 2
FREQ = np.fft.rfftfreq(len(XC))
OKF = (FREQ > 0) & (1 / np.where(FREQ > 0, FREQ, 1) <= 520)    # ciclos de até 10 anos
PERIODOS = 1 / FREQ[OKF][np.argsort(POT[OKF])[::-1][:3]]
print("\nParte 7. Ciclos com mais energia no Ibovespa semanal (sem a tendência): "
      + ", ".join(f"{p:.0f} semanas (cerca de {nr(p / 52, 1)} anos)" for p in PERIODOS))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3.5))
plt.plot((1 / FREQ[OKF]) / 52, POT[OKF] / POT[OKF].max(), color="black")
plt.xlim(0.5, 10)
plt.xlabel("período do ciclo (anos)")
plt.title("Periodograma do Ibovespa semanal")
plt.show()

## Parte 8: screener sazonal no Dow Jones

In [ ]:
ANOS = range(2011, 2026)
INICIOS, DURACOES = np.arange(90), np.arange(25, 61)
F30 = D["Dow30"]


def segmentos(serie, embaralhar=False):
    """Para cada ano, preços (dias corridos) de 5/10 até 150 dias depois; opcionalmente embaralha os retornos."""
    linhas = []
    for a in ANOS:
        base = pd.Timestamp(a, 10, 5)
        s = serie.loc[base - pd.Timedelta(days=7): base + pd.Timedelta(days=150)].dropna()
        if embaralhar:
            r = np.diff(np.log(s.values))
            s = pd.Series(np.exp(np.log(s.values[0]) + np.concatenate([[0], np.cumsum(rng.permutation(r))])), s.index)
        dias = pd.date_range(base, periods=151, freq="D")
        linhas.append(s.reindex(s.index.union(dias)).ffill().loc[dias].values)
    return np.vstack(linhas)


def janelas(seg):
    fim = INICIOS[:, None] + DURACOES[None, :]
    return seg[:, fim] / seg[:, INICIOS][:, :, None] - 1


def melhor(R):
    media = R.mean(axis=0)
    acerto = np.where(media > 0, (R > 0).mean(axis=0), (R < 0).mean(axis=0))
    ok = (acerto >= 0.60) & (np.abs(media) >= 0.03)
    if not ok.any():
        return None
    i, k = np.unravel_index(np.where(ok, np.abs(media), -1).argmax(), media.shape)
    return acerto[i, k], media[i, k], i, k


ACHADOS, DENTRO, FORA, BASE = 0, [], [], []
for t in DOW:
    R = janelas(segmentos(F30[t]))
    ACHADOS += melhor(R) is not None
    m = melhor(R[:10])                                          # escolhe a janela com 2011 a 2020...
    if m is not None:
        ac, med, i, k = m
        r = R[10:, i, k]                                        # ...e confere em 2021 a 2025
        DENTRO.append(ac)
        FORA.append(((r > 0) if med > 0 else (r < 0)).mean())
    BASE.append((R[10:] > 0).mean())
FALSOS = [sum(melhor(janelas(segmentos(F30[t], True))) is not None for t in DOW) for _ in range(10)]
print(f"\nParte 8. Screener sazonal no Dow (30 ações, 15 anos, início entre 5/10 e 3/1, 25 a 60 dias, acerto de 60%, média"
      " de 3%)")
print(f"   ações com pelo menos um 'padrão': {ACHADOS} de 30 | com os pregões embaralhados: {nr(np.mean(FALSOS), 0)} de 30 em"
      " média (10 sorteios)")
print(f"   janela escolhida em 2011 a 2020: acerto de {pc(np.mean(DENTRO), 0)} na escolha e de {pc(np.mean(FORA), 0)} em"
      f" 2021 a 2025 ({len(FORA)} ações) | qualquer janela comprada em 2021 a 2025: {pc(np.mean(BASE), 0)}")

## Parte 9: a queda dentro do ano

In [ ]:
def quedas(serie, ini, fim):
    q = {}
    for a in range(ini, fim + 1):
        s = serie.loc[str(a)]
        q[a] = (s / s.cummax() - 1).min()
    return pd.Series(q)


Q_IB, Q_SP = quedas(IB, 1996, 2025), quedas(SPX, 1996, 2025)
print(f"\nParte 9. Queda máxima dentro de cada ano (1996 a 2025): Ibovespa caiu 10% ou mais em {(Q_IB <= -0.10).sum()} de"
      f" {len(Q_IB)} anos (mediana {pc(Q_IB.median(), 0)}); S&P 500 em {(Q_SP <= -0.10).sum()} de {len(Q_SP)} (mediana"
      f" {pc(Q_SP.median(), 0)})")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a Parte 7. Mostre como `np.polyfit(..., 2)` tira a tendência do Ibovespa semanal, para que serve `np.hanning`, o que `np.fft.rfft` calcula e por que `OKF` limita os ciclos a 520 semanas." Confira com a Parte 7, que imprime os três ciclos com mais energia: 331, 184 e 166 semanas.
2. **Adaptar:** "Acrescente [ATIVO] ao dicionário da célula de abertura, com o ticker do Yahoo Finance e a data inicial. Numa célula nova, depois da Parte 2, repita a conta da Parte 2 para [ATIVO], só com anos fechados e com um gerador próprio, `np.random.default_rng(42)`. Não mude a conta do p-valor nem os 5.000 sorteios." Antes de olhar o ativo novo, rode o notebook do início, em ordem, e confira se a Parte 2 continua com maio a -0,5% e p-valor de 0,21, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque a média pela mediana. Algum mês muda de posição?
2. Na Parte 3, desconte uma taxa de juros fixa de 10% ao ano nos meses fora da bolsa. A estratégia dos seis meses passa o comprar e segurar?
3. Na Parte 8, exija acerto de 80%. Quantas ações ainda têm um 'padrão', com e sem os pregões embaralhados?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.